# Pairwise distances and nearest neighbours with nested vmap

Write the distance between **two vectors**, then turn it into a full distance matrix with two nested `jax.vmap` calls: the inner one maps over the rows of `Y`, the outer one over the rows of `X`.

Write `pairwise(X, Y, metric)`:

- `X`: `(N, D)`; `Y`: `(M, D)`; float32.
- `metric`: Python string, one of
  - `'euclidean'`: $\lVert x - y \rVert_2$
  - `'manhattan'`: $\sum_i |x_i - y_i|$
  - `'cosine'`: $1 - \dfrac{x \cdot y}{\lVert x \rVert\, \lVert y \rVert}$. If either vector is all zeros, the cosine similarity is defined as 0, so the distance is 1.

Return `(dist, nearest)`:

- `dist`: `(N, M)` float32 with `dist[i, j]` the distance between `X[i]` and `Y[j]`.
- `nearest`: `(N,)` int, the index of the closest row of `Y` for each row of `X`. Ties go to the **lowest** index.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def pairwise(X, Y, metric):
    """Return (dist (N, M), nearest (N,))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[0.0, 0.0], [3.0, 4.0]], "dtype": "float64"}, {"__nd__": [[3.0, 4.0], [1.0, 0.0]], "dtype": "float64"}, "euclidean"], "e": {"__tup__": [{"__nd__": [[5.0, 1.0], [0.0, 4.4721360206604]], "dtype": "float32"}, {"__nd__": [1, 0], "dtype": "int32"}]}}, {"a": [{"__nd__": [[1.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[0.0, 2.0], [-1.0, 0.0], [3.0, 0.0]], "dtype": "float64"}, "cosine"], "e": {"__tup__": [{"__nd__": [[1.0, 2.0, 0.0], [1.0, 1.0, 1.0]], "dtype": "float32"}, {"__nd__": [2, 0], "dtype": "int32"}]}}, {"a": [{"__nd__": [[1.0, 2.0, 3.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0], [1.0, 2.0, 4.0]], "dtype": "float64"}, "manhattan"], "e": {"__tup__": [{"__nd__": [[6.0, 1.0]], "dtype": "float32"}, {"__nd__": [1], "dtype": "int32"}]}}, {"a": [{"__nd__": [[1.3932604931912123, 0.5328440162191196, 1.660637540567844], [0.6342201141568383, -0.17774108348933784, -1.5244078842009887], [-1.5605397007962647, 0.20478747676598744, 0.6909785837975579], [1.4433055912728368, -0.6133016958329327, 0.07777395103816398]], "dtype": "float64"}, {"__nd__": [[-1.4308730228590871, -0.9365477163197146, 0.3939383568003809], [-0.5240866279019979, 0.525616204517509, 0.8073236234093464], [-1.4435313882996355, 1.017063788892187, -0.5956486475349956], [2.0940849859026374, 0.7286889415434207, -0.5389894305810751], [0.20148779022628246, 0.21497791867789423, 0.3295181094194298], [-1.4308730228590871, -0.9365477163197146, 0.3939383568003809], [-0.5240866279019979, 0.525616204517509, 0.8073236234093464], [-1.4435313882996355, 1.017063788892187, -0.5956486475349956], [2.0940849859026374, 0.7286889415434207, -0.5389894305810751], [0.20148779022628246, 0.21497791867789423, 0.3295181094194298]], "dtype": "float64"}, "euclidean"], "e": {"__tup__": [{"__nd__": [[3.426276206970215, 2.098670244216919, 3.6568682193756104, 2.316866159439087, 1.8147286176681519, 3.426276206970215, 2.098670244216919, 3.6568682193756104, 2.316866159439087, 1.8147286176681519], [2.918980836868286, 2.696916341781616, 2.570448160171509, 1.9808762073516846, 1.9438433647155762, 2.918980836868286, 2.696916341781616, 2.570448160171509, 1.9808762073516846, 1.9438433647155762], [1.1864620447158813, 1.0911929607391357, 1.5260711908340454, 3.891474723815918, 1.7987490892410278, 1.1864620447158813, 1.0911929607391357, 1.5260711908340454, 3.891474723815918, 1.7987490892410278], [2.9095277786254883, 2.3874690532684326, 3.3831076622009277, 1.613954782485962, 1.5137813091278076, 2.9095277786254883, 2.3874690532684326, 3.3831076622009277, 1.613954782485962, 1.5137813091278076]], "dtype": "float32"}, {"__nd__": [4, 4, 1, 4], "dtype": "int32"}]}}, {"a": [{"__nd__": [[-0.8129783500203097, -0.42233336776846886, -2.279448848941372, 0.7388424545328427, -0.7226053236757057, -0.0726952638891593, 0.44573296792477696, 0.33908848192705], [-1.314231851947895, 1.518167748789026, 0.7573140535776205, 0.500135002991362, -0.6595974890042613, -1.604275539508039, -1.6301579857228963, -0.2084186249328172], [0.16067141270164298, -0.7033605256162605, 1.8294451175597746, 0.301141363409508, 0.9985196810136125, 0.9065691321231881, -1.0704716547313502, 0.9938796078896032], [0.45020245760329053, -1.268629774274663, 0.7234396764533373, 1.0719754356695892, 0.9765949301257374, 1.0518536912100145, -1.410116363735111, -0.24749669441881675], [-0.5457172094037283, 0.054290758129795294, 0.10624998985929954, 1.1260779338878808, 0.7521507767954232, -0.8289487151189747, 0.9014636363022173, 1.6261274490886326], [0.3063559833452629, -1.5056171465064125, -0.6447046126057875, -0.17950826262875338, -1.623352218247181, -0.13829556178942337, -1.2726643565769213, 0.7926174222361886]], "dtype": "float64"}, {"__nd__": [[0.6843771013059606, -0.07646868114989751, -1.2547871718792771, 0.2892103756592987, -0.637470957995064, -0.6877455131350327, 1.1100333101622353, -0.7571494341542212], [-1.376811172406851, 0.7903706911051397, 1.7780291348965471, 1.6092832087165578, 0.7810233860670623, 2.092095347333082, -0.3555269690997188, 2.175355856954505], [2.0984629871312794, -0.34232473779069145, 0.41314577130707864, -1.2868222290420046, -1.026217133402354, 0.5658305979005168, -1.1636012482496745, 0.21487308247743178], [0.46271614740961764, 0.140062517700594, 0.9782996385996884, 1.9572845400255356, 0.19555157182395658, 1.0221838780659158, 1.5142063347162669, -0.4371294946964276], [-0.6492400211935218, -2.413441518597062, 0.45204971802015875, 1.076810389305613, 0.46086742473831777, 2.127234115576448, 1.5577614941377669, 0.8454575963161979], [-0.00892384855406432, -0.8509684185092534, 0.1393736107632248, 0.5317212144155032, -0.03649395726935765, -0.2750942211267223, -0.35509441062106906, 0.7307364273251669], [-1.6800335765838175, -0.28807770896665386, 0.1018816669686754, 0.7264209197923949, 0.7336526217200353, -0.19755499509105373, -0.5546511768058272, -1.1421971668778204]], "dtype": "float64"}, "cosine"], "e": {"__tup__": [{"__nd__": [[0.4483625888824463, 1.1910738945007324, 1.3888440132141113, 1.1137371063232422, 0.8315966129302979, 0.8403223752975464, 0.9007647633552551], [1.273205280303955, 0.8976900577545166, 1.2024208307266235, 1.2962268590927124, 1.653677225112915, 1.0038790702819824, 0.5869780778884888], [1.8497381210327148, 0.31676071882247925, 0.7770622968673706, 0.8275372982025146, 0.6110055446624756, 0.5093018412590027, 0.9475942254066467], [1.4821844100952148, 0.6321205496788025, 0.8222846388816833, 0.7429919838905334, 0.5820879340171814, 0.5241954326629639, 0.6888664960861206], [1.059080958366394, 0.5442142486572266, 1.5953574180603027, 0.7255768179893494, 0.7116388082504272, 0.5072773098945618, 0.9851760268211365], [0.9673534631729126, 1.2053816318511963, 0.4798910617828369, 1.464923620223999, 0.9455223679542542, 0.3975386619567871, 1.2488583326339722]], "dtype": "float32"}, {"__nd__": [0, 6, 1, 5, 5, 5], "dtype": "int32"}]}}, {"a": [{"__nd__": [[1.1985130309494931, 0.6170455235360233, 0.2371163330472784, -0.69613704517196, -1.0973677991032382], [1.2012952295910957, 0.020417067655769013, 0.5051564226621673, 1.8359317541030848, -1.5018198983591617], [-0.7438716513741757, 0.4010504063562215, 0.6580545227008587, -1.0111340115003025, -0.04821792700186009]], "dtype": "float64"}, {"__nd__": [[-0.5738697728512288, -0.548525471133838, 0.4167379121600267, 0.22900954363797144, 1.2110986214979738], [0.6385901114287856, 1.5287163665093464, -0.7903888958165145, 0.4405798706237835, -1.4002895825921144]], "dtype": "float64"}, "manhattan"], "e": {"__tup__": [{"__nd__": [[6.3511881828308105, 3.938737392425537], [6.75236701965332, 4.863431930541992], [3.860354423522949, 6.762356758117676]], "dtype": "float32"}, {"__nd__": [1, 1, 0], "dtype": "int32"}]}}, {"a": [{"__nd__": [[2.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [1.0, 1.0], [1.0, -1.0]], "dtype": "float64"}, "manhattan"], "e": {"__tup__": [{"__nd__": [[2.0, 2.0, 2.0], [0.0, 2.0, 2.0]], "dtype": "float32"}, {"__nd__": [0, 0], "dtype": "int32"}]}}, {"a": [{"__nd__": [[-16.474457351747425, -13.805242632249053, 7.2770504629754305, 2.127176699125485], [-16.312605066944066, 13.174603283232077, 9.747344198069346, -1.8016573421487803], [-2.4211181807825635, -0.11861858734231177, -12.234693550745348, 23.512144906486267], [-4.705912418400215, -1.965183331051471, -3.285763832906877, -6.737297547794211], [-9.136252632592344, 3.3111225984483763, -12.441186317884279, -10.5277979061275]], "dtype": "float64"}, {"__nd__": [[-8.169077497979886, 8.004215366408655, -10.048895061623913, -10.055925231844835], [3.192914884267971, -11.442260552989964, -3.3579885178697255, 8.835270573131897], [4.0319154852707335, 0.21310898823727664, 15.186179629799257, -4.297840143908866], [-6.199193994724064, -17.718891045079513, 0.21381322491167437, 6.2042078182873786], [-6.825700829758153, 18.733830622140292, 10.091932832333402, 9.130193047967209], [-12.213808688698624, 2.806476200148147, 3.5487194073794703, -20.59977724204916], [-0.37651399631622073, -2.555685832588823, -17.551650799567156, 4.79395734165729], [20.41800016544976, 7.389231871807743, -8.67567623294184, -2.7166666245550926], [11.127446451013581, -8.480567224366947, 3.444554227590388, 0.221629614372186]], "dtype": "float64"}, "euclidean"], "e": {"__tup__": [{"__nd__": [[31.515838623046875, 23.462560653686523, 26.848848342895508, 13.689746856689453, 34.76860427856445, 28.714406967163086, 31.769046783447266, 45.69695281982422, 28.434772491455078], [23.51752281188965, 35.655982971191406, 24.85380744934082, 34.80906677246094, 15.509044647216797, 22.71762466430664, 35.918495178222656, 41.50726318359375, 35.57698059082031], [35.08008575439453, 21.306072235107422, 39.58599853515625, 27.902929306030273, 32.86539077758789, 47.9523811340332, 19.717004776000977, 35.757659912109375, 32.27650451660156], [12.966713905334473, 19.867475509643555, 20.694400787353516, 20.739788055419922, 29.38848114013672, 17.832996368408203, 18.856769561767578, 27.639331817626953, 19.668994903564453], [5.3764543533325195, 28.75940704345703, 31.386016845703125, 29.84956932067871, 33.725013732910156, 19.15327262878418, 19.287961959838867, 31.068918228149414, 30.291391372680664]], "dtype": "float32"}, {"__nd__": [3, 4, 6, 0, 0], "dtype": "int32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(pairwise, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")